<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/openai_function_calling_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI API Function Calling 예제

## Chat Completion API Reference : https://platform.openai.com/docs/api-reference/chat/create

## Reference : https://cookbook.openai.com/examples/how_to_call_functions_with_chat_models

In [1]:
!pip install scipy
!pip install tenacity
!pip install tiktoken
!pip install termcolor
!pip install openai
!pip install requests

In [13]:
import json
import requests
from openai import OpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt
from termcolor import colored

# 1. API 키 변수 설정
OPENAI_KEY = "Input Your API Key"

# 2. 클라이언트 생성 (변수 사용)
client = OpenAI(api_key=OPENAI_KEY)

GPT_MODEL = "gpt-4o-mini"

## OpenAI API Key 설정

In [14]:
@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, tools=None, tool_choice=None, model=GPT_MODEL): #tools = Function 사용, tool_choice = 특정 Function을 강제하는 코드
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + OPENAI_KEY,
    }
    json_data = {"model": model, "messages": messages}
    if tools is not None:
        json_data.update({"tools": tools})
    if tool_choice is not None:
        json_data.update({"tool_choice": tool_choice})
    try:
        response = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers=headers,
            json=json_data,
        )
        return response
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e

# 날씨 관련 함수 function calling 예제

In [21]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather", #Function 이름
            "description": "Get the current weather", #현재 날씨를 받아온는 Function이라는 설
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA", #어느 지역의 날씨를 받아올 것인지에 대한 설정
                    },
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"], #화씨 섭씨 설정
                        "description": "The temperature unit to use. Infer this from the users location.",
                    },
                },
                "required": ["location", "format"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_n_day_weather_forecast", #Function 이름
            "description": "Get an N-day weather forecast", #인자 값으로 받은 것을 며칠 뒤에 날씨 예측
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use. Infer this from the users location.",
                    },
                    "num_days": {
                        "type": "integer",
                        "description": "The number of days to forecast",
                    }
                },
                "required": ["location", "format", "num_days"]
            },
        }
    },
]

In [24]:
messages = []
messages.append({"role": "system", "content": "함수에 값을 입력할 때 가정을 하지 마세요. 사용자 요청이 모호하면 명확히 하라고 요청하세요."})
messages.append({"role": "user", "content": "다음 며칠 동안 스코틀랜드 글래스고의 날씨가 어때?"})
chat_response = chat_completion_request(
    messages, tools=tools
)
print(chat_response.json())
assistant_message = chat_response.json()["choices"][0]["message"]
messages.append(assistant_message)
assistant_message

{'id': 'chatcmpl-D0M5tRL5AKXqnOXAGiWXqU5fQ8WKu', 'object': 'chat.completion', 'created': 1768977301, 'model': 'gpt-4o-mini-2024-07-18', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '글래스고의 날씨 예보를 몇 일 동안 알고 싶으신가요? 며칠 간의 예보를 원하시는지 알려주세요. (예: 3일, 5일 등)', 'refusal': None, 'annotations': []}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 204, 'completion_tokens': 48, 'total_tokens': 252, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': 'fp_29330a9688'}


{'role': 'assistant',
 'content': '글래스고의 날씨 예보를 몇 일 동안 알고 싶으신가요? 며칠 간의 예보를 원하시는지 알려주세요. (예: 3일, 5일 등)',
 'refusal': None,
 'annotations': []}

In [19]:
messages.append({"role": "user", "content": "저는 스코틀랜드 글래스고에 날씨를 알고싶어요."})
chat_response = chat_completion_request(
    messages, tools=tools
)
print(chat_response.json())
assistant_message = chat_response.json()["choices"][0]["message"]
messages.append(assistant_message)
assistant_message

{'id': 'chatcmpl-D0LyTHCnKvXCKWrOxwahWqOVp0API', 'object': 'chat.completion', 'created': 1768976841, 'model': 'gpt-4o-mini-2024-07-18', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '스코틀랜드 글래스고의 날씨 예보를 확인하기 위해 며칠 동안의 예보를 원하는지 알려주세요. 예를 들어, 3일, 5일, 7일 중 어느 기간이 좋으신가요?', 'refusal': None, 'annotations': []}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 286, 'completion_tokens': 56, 'total_tokens': 342, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': 'fp_29330a9688'}


{'role': 'assistant',
 'content': '스코틀랜드 글래스고의 날씨 예보를 확인하기 위해 며칠 동안의 예보를 원하는지 알려주세요. 예를 들어, 3일, 5일, 7일 중 어느 기간이 좋으신가요?',
 'refusal': None,
 'annotations': []}

In [20]:
messages = []
messages.append({"role": "system", "content": "함수에 값을 입력할 때 가정을 하지 마세요. 사용자 요청이 모호하면 명확히 하라고 요청하세요."})
messages.append({"role": "user", "content": "다음 며칠 동안 스코틀랜드 글래스고의 날씨가 어떨까요?"})
chat_response = chat_completion_request(
    messages, tools=tools
)
print(chat_response.json())
assistant_message = chat_response.json()["choices"][0]["message"]
messages.append(assistant_message)
assistant_message

{'id': 'chatcmpl-D0Lyt61JE9br5ddrtW4YmEF3Oi265', 'object': 'chat.completion', 'created': 1768976867, 'model': 'gpt-4o-mini-2024-07-18', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '글래스고의 날씨 예보를 제공하기 위해 몇 일 동안의 예보를 알고 싶으신가요? 예를 들어, 3일, 5일, 7일 중에서 선택할 수 있습니다. 몇 일의 예보를 원하시는지 알려주세요.', 'refusal': None, 'annotations': []}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 205, 'completion_tokens': 63, 'total_tokens': 268, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': 'fp_29330a9688'}


{'role': 'assistant',
 'content': '글래스고의 날씨 예보를 제공하기 위해 몇 일 동안의 예보를 알고 싶으신가요? 예를 들어, 3일, 5일, 7일 중에서 선택할 수 있습니다. 몇 일의 예보를 원하시는지 알려주세요.',
 'refusal': None,
 'annotations': []}

## 특정 함수나 아무 함수도 사용하지 않도록 강제하는 방법(Forcing the use of specific functions or no function)

In [25]:
# in this cell we force the model to use get_n_day_weather_forecast
messages = []
messages.append({"role": "system", "content": "함수에 값을 입력할 때 가정을 하지 마세요. 사용자 요청이 모호하면 명확히 하라고 요청하세요."})
messages.append({"role": "user", "content": "캐나다 토론토의 날씨를 알려주세요"})
chat_response = chat_completion_request(
    messages, tools=tools, tool_choice={"type": "function", "function": {"name": "get_n_day_weather_forecast"}} #강제하고 싶은 함수 설정
)
chat_response.json()["choices"][0]["message"]

{'role': 'assistant',
 'content': None,
 'tool_calls': [{'id': 'call_A2kFArb6jGYntSyhq4S2qfrY',
   'type': 'function',
   'function': {'name': 'get_n_day_weather_forecast',
    'arguments': '{"location":"Toronto, Canada","format":"celsius","num_days":1}'}}],
 'refusal': None,
 'annotations': []}

In [26]:
# if we don't force the model to use get_n_day_weather_forecast it may not
messages = []
messages.append({"role": "system", "content": "함수에 값을 입력할 때 가정을 하지 마세요. 사용자 요청이 모호하면 명확히 하라고 요청하세요."})
messages.append({"role": "user", "content": "캐나다 토론토의 날씨를 알려주세요"})
chat_response = chat_completion_request(
    messages, tools=tools
)
chat_response.json()["choices"][0]["message"]

{'role': 'assistant',
 'content': '현재 날짜와 기온 단위 (섭씨 또는 화씨)를 알려주시면 토론토의 날씨를 확인해드릴 수 있습니다. 어떤 단위를 원하시나요?',
 'refusal': None,
 'annotations': []}

In [27]:
messages = []
messages.append({"role": "system", "content": "함수에 값을 입력할 때 가정을 하지 마세요. 사용자 요청이 모호하면 명확히 하라고 요청하세요."})
messages.append({"role": "user", "content": "캐나다 토론토의 현재 날씨를 섭씨로 알려주세요."})
chat_response = chat_completion_request(
    messages, tools=tools, tool_choice="none"
)
print(chat_response.json())
chat_response.json()["choices"][0]["message"]

{'id': 'chatcmpl-D0M7Sb0HGyr01Aboc0rpSMtdhsyLv', 'object': 'chat.completion', 'created': 1768977398, 'model': 'gpt-4o-mini-2024-07-18', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '토론토의 현재 날씨를 확인하겠습니다. 잠시만 기다려 주세요.', 'refusal': None, 'annotations': []}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 201, 'completion_tokens': 19, 'total_tokens': 220, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': 'fp_29330a9688'}


{'role': 'assistant',
 'content': '토론토의 현재 날씨를 확인하겠습니다. 잠시만 기다려 주세요.',
 'refusal': None,
 'annotations': []}

## Parallel Function Calling

In [29]:
#한번 호출했지만 두번의 병렬적으로 함수 호출을 하였다
messages = []
messages.append({"role": "system", "content": "함수에 값을 입력할 때 가정을 하지 마세요. 사용자 요청이 모호하면 명확히 하라고 요청하세요."})
messages.append({"role": "user", "content": "다음 4일 동안 샌프란시스코와 글래스고의 날씨가 어떨까요?(섭씨로)"})
chat_response = chat_completion_request(
    messages, tools=tools, model='gpt-4o-mini'
)
print(chat_response.json())
assistant_message = chat_response.json()["choices"][0]["message"]['tool_calls']
assistant_message

{'id': 'chatcmpl-D0MA9uO6pyJVc2Ay6IXZYaivOQgXs', 'object': 'chat.completion', 'created': 1768977565, 'model': 'gpt-4o-mini-2024-07-18', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call_CJppZ0r46SMpTHyS8Dtk5O5L', 'type': 'function', 'function': {'name': 'get_n_day_weather_forecast', 'arguments': '{"location": "San Francisco, CA", "format": "celsius", "num_days": 4}'}}, {'id': 'call_HujvmCCu4xZs7xrPHnevb4Qm', 'type': 'function', 'function': {'name': 'get_n_day_weather_forecast', 'arguments': '{"location": "Glasgow, Scotland", "format": "celsius", "num_days": 4}'}}], 'refusal': None, 'annotations': []}, 'logprobs': None, 'finish_reason': 'tool_calls'}], 'usage': {'prompt_tokens': 213, 'completion_tokens': 79, 'total_tokens': 292, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'se

[{'id': 'call_CJppZ0r46SMpTHyS8Dtk5O5L',
  'type': 'function',
  'function': {'name': 'get_n_day_weather_forecast',
   'arguments': '{"location": "San Francisco, CA", "format": "celsius", "num_days": 4}'}},
 {'id': 'call_HujvmCCu4xZs7xrPHnevb4Qm',
  'type': 'function',
  'function': {'name': 'get_n_day_weather_forecast',
   'arguments': '{"location": "Glasgow, Scotland", "format": "celsius", "num_days": 4}'}}]